In [23]:
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from configparser import ConfigParser
from sqlalchemy import create_engine
import cartopy
import pyproj
import movingpandas as mpd
from shapely.ops import transform
import osmnx

### Configuration

Read parameter from the configuration file, it should be called `setting.ini` and after it connect to the database

In [2]:
config = ConfigParser()
config.read("setting.ini")
dbsett = config["eurodeer_db"]

In [3]:
db_connection_url = "postgresql://{us}:{pas}@{host}:{port}/{db}".format(us=dbsett['user'],
                                                                      pas=dbsett['password'],
                                                                      host=dbsett['host'],
                                                                      port=dbsett['port'],
                                                                      db=dbsett["db"]
                                                                     )
con = create_engine(db_connection_url)

## Get data

Get all animals with death date

In [8]:
animals_death_query = "select animals_id, death_date, death_time, study_areas_id, mortality_code from main.animals where death_date is not null order by animals_id"
animals_death_df = pd.read_sql(all_death_query, con)

In [9]:
print("Number of animals: {}".format(len(all_death_df)))
all_death_df.head()

Number of animals: 949


,animals_id,death_date,death_time,study_areas_id,mortality_code
0,1,2006-11-30,20:03:13+00:00,2,5
1,2,2008-03-06,10:00:00+00:00,2,5
2,3,2008-09-09,None,2,5
3,4,2007-06-07,18:03:06+00:00,2,7
4,5,2008-05-23,14:01:49+00:00,2,7


Get the position of the last point recorded by gps or vhf of animals with death date, only if longitude and latitude are present.

In [15]:
#this query return only gps data
#animals_query = "select distinct on (g.animals_id) g.acquisition_time, g.gps_data_animals_id, g.latitude, g.longitude, q.*, g.geom from (select a.animals_id, a.death_date, a.death_time, a.study_areas_id from main.animals as a where a.death_date is not null order by a.animals_id) as q, main.gps_data_animals as g where q.animals_id=g.animals_id and q.death_date = date(g.acquisition_time) and (g.latitude is not null and g.longitude is not null)  order by g.animals_id asc, acquisition_time desc;"
animals_query = "select * from (select distinct on (g.animals_id) g.acquisition_time, g.gps_data_animals_id as data_animals_id, g.latitude, g.longitude, q.*, g.geom from (select a.animals_id, a.death_date, a.death_time, a.study_areas_id, a.mortality_code from main.animals as a where a.death_date is not null order by a.animals_id) as q, main.gps_data_animals as g where q.animals_id=g.animals_id UNION select distinct on (g.animals_id) g.acquisition_time, g.vhf_data_animals_id as data_animals_id, g.latitude, g.longitude, q.*, g.geom from (select a.animals_id, a.death_date, a.death_time, a.study_areas_id, a.mortality_code from main.animals as a where a.death_date is not null order by a.animals_id) as q, main.vhf_data_animals as g where q.animals_id=g.animals_id) as q where geom is not null;"
animals_df = gpd.GeoDataFrame.from_postgis(animals_query, con)

Let's see the first rows of the data

In [16]:
print("Number of animals: {}".format(len(animals_df)))
animals_df.head()

Number of animals: 607


,acquisition_time,data_animals_id,latitude,longitude,animals_id,death_date,death_time,study_areas_id,mortality_code,geom
0,2005-09-30 18:02:52+00:00,108658,49.158388,13.269931,9,2006-05-31,18:01:36+00:00,2,8,POINT (13.26993 49.15839)
1,2013-07-23 11:02:32+00:00,10161643,43.335092,1.357835,4810,2017-08-31,None,8,7,POINT (1.35783 43.33509)
2,2008-03-13 10:20:55+00:00,10950122,43.281660,0.873586,5056,2008-08-03,None,8,5,POINT (0.87359 43.28166)
3,2013-03-27 08:00:53+00:00,10440508,43.281577,0.900265,4807,2013-11-16,None,8,7,POINT (0.90026 43.28158)
4,2014-02-15 11:00:44+00:00,6706307,46.045275,10.692035,2449,2014-06-12,00:37:00+00:00,24,3,POINT (10.69204 46.04528)


In [17]:
animals_df.hvplot.points(x='longitude', y='latitude', tiles='OSM', geo=True, width=400, height=400)

:Overlay
   .Tiles.I  :Tiles   [x,y]
   .Points.I :Points   [longitude,latitude]

Filter animals with same aquisition time year and death date year

In [45]:
filtered_values_year = np.where(animals_df['acquisition_time'].dt.year == pd.to_datetime(animals_df['death_date']).dt.year)
print(filtered_values_year)

animals_same_year = animals_df.loc[filtered_values_year]
print("Number of animals: {}".format(len(animals_same_year)))
animals_same_year.head()

(array([  2,   3,   4,   5,   8,   9,  10,  12,  14,  16,  19,  20,  22,
        23,  25,  26,  28,  30,  31,  33,  34,  35,  37,  39,  40,  41,
        42,  43,  44,  45,  48,  49,  50,  51,  52,  53,  60,  63,  64,
        65,  66,  68,  69,  70,  71,  74,  76,  77,  78,  79,  80,  81,
        82,  83,  89,  90,  91,  92,  93,  96,  99, 101, 107, 108, 109,
       112, 113, 114, 115, 117, 125, 126, 127, 129, 130, 131, 132, 139,
       140, 142, 144, 145, 146, 147, 151, 154, 155, 156, 157, 158, 160,
       161, 163, 164, 165, 167, 168, 169, 170, 177, 178, 180, 181, 182,
       183, 185, 186, 187, 188, 190, 191, 192, 197, 198, 200, 201, 203,
       205, 206, 207, 209, 210, 212, 217, 218, 219, 220, 221, 223, 225,
       227, 228, 229, 231, 234, 235, 236, 237, 240, 243, 246, 247, 250,
       251, 252, 253, 255, 259, 260, 264, 266, 267, 270, 271, 273, 274,
       276, 277, 278, 280, 282, 286, 287, 288, 289, 290, 291, 293, 295,
       296, 300, 305, 306, 307, 308, 309, 311, 312, 313, 314, 3

,acquisition_time,data_animals_id,latitude,longitude,animals_id,death_date,death_time,study_areas_id,mortality_code,geom
2,2008-03-13 10:20:55+00:00,10950122,43.281660,0.873586,5056,2008-08-03,None,8,5,POINT (0.87359 43.28166)
3,2013-03-27 08:00:53+00:00,10440508,43.281577,0.900265,4807,2013-11-16,None,8,7,POINT (0.90026 43.28158)
4,2014-02-15 11:00:44+00:00,6706307,46.045275,10.692035,2449,2014-06-12,00:37:00+00:00,24,3,POINT (10.69204 46.04528)
5,2008-02-07 09:42:23+00:00,1823196,48.882286,13.551329,10,2008-12-14,06:00:48+00:00,2,5,POINT (13.55133 48.88229)
8,2012-02-23 07:00:31+00:00,4741462,47.886520,8.721234,2059,2012-07-16,08:30:00+00:00,31,5,POINT (8.72123 47.88652)


In [51]:
filtered_values_month = np.where((animals_df['acquisition_time'].dt.year == pd.to_datetime(animals_df['death_date']).dt.year) & (animals_same_year['acquisition_time'].dt.month == pd.to_datetime(animals_same_year['death_date']).dt.month))
animals_same_month = animals_df.loc[filtered_values_month]
print("Number of animals: {}".format(len(animals_same_month)))
animals_same_month.head()

Number of animals: 95


,acquisition_time,data_animals_id,latitude,longitude,animals_id,death_date,death_time,study_areas_id,mortality_code,geom
9,2009-04-14 18:01:16+00:00,10160014,43.390620,1.002357,4891,2009-04-24,None,8,8,POINT (1.00236 43.39062)
14,1986-02-20 12:30:20+00:00,71100,48.920653,13.394420,2413,1986-02-21,00:00:00+00:00,2,8,POINT (13.39442 48.92065)
23,2010-12-23 15:00:14+00:00,6314569,58.136747,12.407001,2322,2010-12-28,00:00:00+00:00,3,2,POINT (12.40700 58.13675)
28,2008-03-06 19:01:30+00:00,1730838,48.926349,13.450043,1377,2008-03-27,14:01:18+00:00,2,5,POINT (13.45004 48.92635)
35,2009-03-06 12:01:37+00:00,11647599,43.277483,0.935126,5161,2009-03-06,None,8,10,POINT (0.93513 43.27748)


We look all the points if they are close to the streets

In [72]:
close_road = []
for index, point in animals_same_month.iterrows():
    try:
        osmdata = osmnx.graph_from_point((point.geom.x, point.geom.y), dist=50, dist_type="bbox", network_type="all_private")
        close_road.append(point)
    except:
        print("No data for animal {}".format(point['animals_id']))

No data for animal 4891
No data for animal 2413
No data for animal 2322
No data for animal 1377
No data for animal 5161
No data for animal 2081
No data for animal 5096
No data for animal 1972
No data for animal 4685
No data for animal 5098
No data for animal 4792
No data for animal 1605
No data for animal 3504
No data for animal 2058
No data for animal 1959
No data for animal 4801
No data for animal 1971
No data for animal 1986
No data for animal 3554
No data for animal 1553
No data for animal 4885
No data for animal 5
No data for animal 1427
No data for animal 8
No data for animal 4907
No data for animal 4795
No data for animal 4689
No data for animal 1407
No data for animal 1389
No data for animal 2303
No data for animal 1544
No data for animal 2031
No data for animal 2435
No data for animal 4701
No data for animal 5070
No data for animal 2399
No data for animal 4862
No data for animal 3509
No data for animal 1548
No data for animal 2311
No data for animal 3516
No data for animal 514

In [73]:
len(close_road)

0